# 02 · Entrenamiento y MLflow Experiments — Adult Income

**Objetivo:** construir el training set desde el Feature Store, entrenar y comparar **5 modelos** distintos (no variaciones de hiperparámetros de uno solo, sino algoritmos diferentes) y registrar parámetros, métricas y artefactos en MLflow.

> Ejecuta primero el notebook 01. Todos los nombres editables están en la primera celda de configuración.


## 1. Configuración

In [0]:
%pip install --quiet databricks-feature-engineering

dbutils.library.restartPython()


In [0]:
import mlflow
import pandas as pd

from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from mlflow.models.signature import infer_signature
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB

# Detectar el catálogo actual
catalog = spark.catalog.currentCatalog()
database = f"{catalog}.mlops_income_course"

income_raw = f"{database}.income_raw"
income_features = f"{database}.income_features"
experiment_name = "/Shared/mlops_income_course"

mlflow.set_experiment(experiment_name)

fe = FeatureEngineeringClient()

print(f"Catálogo detectado : {catalog}")
print(f"Tabla raw          : {income_raw}")
print(f"Feature Table      : {income_features}")
print(f"Experimento        : {experiment_name}")
print("FeatureEngineeringClient inicializado ✓")


## 2. Training set desde Feature Store

Partimos solo de `client_id` y `target`. `FeatureLookup` recupera las seis variables usando la clave.


In [0]:
print(income_features)


In [0]:
# DataFrame base de etiquetas: solo la clave y el target
labels_df = spark.read.table(income_raw).select("client_id", "target")

display(labels_df)


In [0]:
display(spark.sql(f"DESCRIBE HISTORY {income_features}"))


In [0]:
from pyspark.sql.functions import col

def get_latest_delta_version(table_name: str) -> int:
    """Devuelve la versión Delta más reciente de una tabla usando DESCRIBE HISTORY."""
    history = spark.sql(f"DESCRIBE HISTORY {table_name}")
    latest_version = history.orderBy(col("version").desc()).first()["version"]
    return latest_version

# Capturar versiones de las tablas
income_raw_version = get_latest_delta_version(income_raw)
income_features_version = get_latest_delta_version(income_features)

print(f"{income_raw}      → versión {income_raw_version}")
print(f"{income_features} → versión {income_features_version}")


In [0]:
feature_lookups = [
    FeatureLookup(
        table_name=income_features,
        lookup_key="client_id",
    )
]

training_set = fe.create_training_set(
    df=labels_df,
    feature_lookups=feature_lookups,
    label="target",
    exclude_columns=["client_id"],
)

training_df = training_set.load_df()

display(training_df.limit(10))


## 3. División reproducible

`stratify` conserva la proporción de clases (recordar: el dataset está desbalanceado, ~76% / 24%).


In [0]:
# Convertir el training DataFrame a pandas
pdf = training_df.toPandas()

# Crear un MLflow Dataset desde el Spark DataFrame
# (incluye versiones raw y features en el nombre y target como objetivo)
dataset_name = (
    f"income_raw_v{income_raw_version}__income_features_v{income_features_version}"
)

mlflow_dataset = mlflow.data.from_spark(
    df=training_df,
    name=dataset_name,
    targets="target",
)

print(f"MLflow Dataset: {mlflow_dataset.name}")
print(f"  Targets      : {mlflow_dataset.targets}")
print(f"  Digest       : {mlflow_dataset.digest}")
print()

# Separar X e y
feature_cols = [c for c in pdf.columns if c != "target"]
X = pdf[feature_cols]
y = pdf["target"]

# Train/test 80/20 con semilla 42 y estratificación
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Dimensiones")
print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  y_train : {y_train.shape}")
print(f"  y_test  : {y_test.shape}")
print()
print(f"Features ({len(feature_cols)}): {feature_cols}")


## 4. Cinco modelos, una función

Comparamos 5 algoritmos distintos (no hiperparámetros de uno solo — `GridSearchCV` no serviría aquí porque solo conserva el mejor estimador, no permite ver varios modelos comparados). La función evita duplicar código y deja visibles las cuatro piezas de tracking: parámetros, métricas, modelo e input schema.


In [0]:
def train_and_log(model, run_name):
    """Entrena un modelo dentro de un MLflow run y devuelve un resumen."""
    with mlflow.start_run(run_name=run_name) as run:
        # Registrar el dataset de training
        mlflow.log_input(mlflow_dataset, context="training")

        # Registrar nombres y versiones de las tablas como parámetros
        mlflow.log_params({
            "income_raw": income_raw,
            "income_raw_version": income_raw_version,
            "income_features": income_features,
            "income_features_version": income_features_version,
        })

        # Entrenar
        model.fit(X_train, y_train)

        # Predecir
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        # Métricas
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_proba)

        mlflow.log_metrics({"accuracy": acc, "f1": f1, "auc": auc})

        # Firma e input example
        signature = infer_signature(X_train, model.predict(X_train))

        mlflow.sklearn.log_model(
            model,
            artifact_path="model",
            signature=signature,
            input_example=X_train.head(5),
        )

        return {
            "run_id": run.info.run_id,
            "name": run_name,
            "accuracy": acc,
            "f1": f1,
            "auc": auc,
        }


results = []

# 1. Regresión Logística (con escalamiento dentro de Pipeline)
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])
results.append(train_and_log(lr_pipeline, "logistic_regression"))

# 2. Árbol de Decisión
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
results.append(train_and_log(dt_model, "decision_tree"))

# 3. Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
results.append(train_and_log(rf_model, "random_forest"))

# 4. Gradient Boosting (versión rápida de sklearn para datasets medianos/grandes)
gb_model = HistGradientBoostingClassifier(random_state=42)
results.append(train_and_log(gb_model, "gradient_boosting"))

# 5. Naive Bayes
nb_model = GaussianNB()
results.append(train_and_log(nb_model, "naive_bayes"))

results_df = pd.DataFrame(results).sort_values("auc", ascending=False)
display(results_df)


## 5. Comparación desde MLflow

Abre **Experiments → mlops_income_course** para comparar visualmente parámetros, métricas y artefactos de los 5 runs.


In [0]:
from mlflow.entities import ViewType

# Obtener el experimento configurado
exp = mlflow.get_experiment_by_name(experiment_name)

# Buscar runs ordenados por AUC descendente (máximo 10)
runs = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    max_results=10,
    order_by=["metrics.auc DESC"],
)

# Seleccionar y renombrar columnas de interés
comparison_df = runs[[
    "run_id",
    "tags.mlflow.runName",
    "metrics.accuracy",
    "metrics.f1",
    "metrics.auc",
    "params.income_raw_version",
    "params.income_features_version",
]].rename(columns={
    "tags.mlflow.runName": "name",
    "metrics.accuracy": "accuracy",
    "metrics.f1": "f1",
    "metrics.auc": "auc",
    "params.income_raw_version": "income_raw_version",
    "params.income_features_version": "income_features_version",
})

display(comparison_df)


## Cierre

Un **experimento** agrupa runs; cada **run** conserva qué se entrenó, con qué configuración, qué resultado produjo y qué modelo quedó empaquetado. Con 5 algoritmos distintos comparados por AUC (no solo accuracy, por el desbalance de clases), queda evidencia clara para justificar cuál se promueve a `Champion`. Continúa con el notebook 03.
